# 01 · Data, user-level split and tasks

Cleans MATCHA, removes contaminated controls, assigns the single user-level split shared by every
task and model, and builds all task files (the four MATCHA tasks and the sanity-check tasks).

| | |
|---|---|
| Hardware | CPU (a laptop is enough) |
| Time | ~20 min for notebooks 01 and 02 together |
| Needs | MATCHA under its DUA, unpacked as described in `data/README.md` (or `MATCHA_ROOT` set); internet for the public evaluation datasets |
| Writes | `data/processed/`, `data/raw/mentalbert/`, `data/tasks/<task>/` |

> **Data use.** MATCHA is under a Data Use Agreement. These notebooks print counts and scores only, never posts; keep it that way if you add cells, and do not commit notebooks with outputs.

In [ ]:
import sys
from pathlib import Path

# make `matcha_bench` importable from a fresh clone (not needed after `pip install -e .`)
sys.path.insert(0, str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "src"))
from matcha_bench.notebook import check, setup, step

ROOT = setup()  # every step runs from the repository root
print(ROOT)

## 1. Clean MATCHA and split users

`prepare_matcha.py` drops posts under 2 characters and duplicates, removes the 766 "control" users
who also post in disorder communities, and splits users 70/10/20, stratified by cohort (each
disorder, control, and multi-disorder users as their own stratum), with seed 42
(`configs/data.yaml`). A person's posts never sit on both sides of the split.

In [ ]:
step("scripts/prepare_matcha.py")

In [ ]:
import json

stats = json.loads(Path("data/processed/stats.json").read_text(encoding="utf-8"))
check("raw rows", stats["raw_rows"], 1_882_257)
check("posts after cleaning", stats["posts"], 1_507_148)
check("users", stats["users"], 41_747)
check("contaminated controls removed", stats["contaminated_control_users"], 766)
check("multi-disorder users", stats["multi_disorder_users"], 2_677)
check("users per split", stats["users_by_split"], {"train": 29_222, "dev": 4_175, "test": 8_350})

## 2. Check the split against ours

The fingerprint of a split is the SHA-256 of its sorted user IDs. Matching fingerprints mean the
same people are in train, dev and test as in every reported result. (The ID lists themselves stay
under the DUA; `python scripts/split_fingerprint.py --write-ids` writes them locally.)

In [ ]:
import pandas as pd

sys.path.insert(0, str(ROOT / "scripts"))
from split_fingerprint import fingerprint

REPORTED = {
    "train": "400c57e3fd6003a3443feef7d23a9c640e08f5af1ca64808eedc35ff670cde52",
    "dev": "aac84d29a48e2c971b4a63d6b4a264479b19ab7985622de8a3aae674a5a55410",
    "test": "44934b89108b9d7a049711b6a59fdb229fb63e3006ec69e3ddaaa671fc255042",
}
users = pd.read_parquet("data/processed/users.parquet", columns=["user_id", "split"])
for split, sha in REPORTED.items():
    check(f"{split} fingerprint", fingerprint(users.loc[users["split"] == split, "user_id"]), sha)

## 3. Build the tasks

The MATCHA tasks are designed here (the MATCHA paper defines none):

| Task | Level | Labels | Question |
|---|---|---|---|
| `matcha_disorder_post` | post | 7 | which disorder community (Super-Topic) does the post come from? |
| `matcha_disorder_post_masked` | post | 7 | the same, with disorder, nickname and drug names replaced by `某病` / `某药` |
| `matcha_user_binary_masked` | user | 2 | disorder vs control, from homepage posts only, names masked |
| `matcha_user_multiclass_masked` | user | 8 | 7 disorders + control, from homepage posts only, names masked |

`disorder_post` training is capped at 5,000 posts per disorder and its test set at 20,000 posts
(stratified). The sanity-check tasks are the Chinese MentalBERT paper's public evaluation sets
(Suicide = SOS-HL-1K, Cognitive = SocialCD-3K, Waimai, EWECT); the RoBERTa paper's Reddit task is
built in notebook 05.

In [ ]:
step("scripts/download_mentalbert_data.py")
step("scripts/build_tasks.py")

In [ ]:
REPORTED_SIZES = {
    "matcha_disorder_post": {"train": 23_109, "dev": 14_934, "test": 20_000},
    "matcha_disorder_post_masked": {"train": 23_109, "dev": 14_934, "test": 20_000},
    "matcha_user_binary_masked": {"train": 2_392, "dev": 358, "test": 690},
    "matcha_user_multiclass_masked": {"train": 6_919, "dev": 1_275, "test": 2_105},
    "suicide": {"train": 799, "dev": 200, "test": 250},
    "cognitive": {"train": 2_180, "dev": 545, "test": 682},
    "waimai": {"train": 9_589, "dev": 1_199, "test": 1_199},
    "ewect_epidemic": {"train": 8_592, "dev": 2_000, "test": 3_000},
    "ewect_usual": {"train": 27_730, "dev": 2_000, "test": 5_000},
}
for name, sizes in REPORTED_SIZES.items():
    meta = json.loads(Path(f"data/tasks/{name}/meta.json").read_text(encoding="utf-8"))
    check(name, meta["sizes"], sizes)

Next: `02_lexicon.ipynb`.